# jA_01 - Near-domain partition (JURNAL IJCITSM, Konfig A)

**Konteks:** notebook ini milik **paper JURNAL** (extended dari ISRITI). Berbeda dari
survei cross-domain (`s01`, 9 fitur CIC-vs-UNSW), di sini kedua klien memakai
**ekstraktor yang sama (CICFlowMeter)** sehingga ruang fitur irisannya **penuh
(~76 fitur)**, bukan 9. Inilah sisi **near-domain** (jarak domain dekat) yang
menjadi material BARU jurnal.

**Dua klien:**
- **Client-CIC2017** — CIC-IDS2017 (CICFlowMeter).
- **Client-CIC2018** — CSE-CIC-IDS2018 (CICFlowMeter).

**Penyelarasan fitur = IRISAN nama kolom** (manual name alignment biasa, bukan SFM),
memakai kamus `ALIAS` 2018<->2017 yang sudah terbukti di Paper 4 XAI
(`unswnb-15/notebook_xai/01_preprocessing.ipynb`).

**Output:** `journal_out/` berisi `cic2017_{train,test}.npz`, `cic2018_{train,test}.npz`
(key `X`,`y`,`feats` — kompatibel node EC2), `neardomain_stats.csv`, dan
`neardomain_schema.json`. Di-upload ke S3 prefix BARU `federated_journal/neardomain/`
agar tak bercampur dengan data survei (`federated_survey/`).

**Label biner:** Benign/normal = 0, selain itu = 1.

> Jalankan di SageMaker (tempat CSV 2017/2018 tersedia via S3). JANGAN mengarang angka.

## 0. Setup + KONFIG S3/path

In [ ]:
import importlib, sys, subprocess
for m in ('pandas','numpy','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, glob, json, pickle, gc, warnings, zipfile
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

# ---- S3 ----
S3_BUCKET = os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
REGION    = 'ap-southeast-1'
S3_RAW_CIC2017 = 's3://%s/datasets/CICD2017/' % S3_BUCKET   # CSV CICFlowMeter 2017
S3_RAW_CIC2018 = 's3://%s/datasets/CICD2018/' % S3_BUCKET   # CSV CICFlowMeter 2018
S3_PREFIX = 'federated_journal/neardomain'                  # prefix BARU (jurnal)

# ---- folder lokal di SageMaker ----
DATA_DIR='./data'; CIC2017_DIR=os.path.join(DATA_DIR,'cic2017'); CIC2018_DIR=os.path.join(DATA_DIR,'cic2018')
OUTDIR='journal_out'
for d in (CIC2017_DIR, CIC2018_DIR, OUTDIR): os.makedirs(d, exist_ok=True)
CIC2017_GLOB=os.path.join(CIC2017_DIR,'**','*.csv'); CIC2018_GLOB=os.path.join(CIC2018_DIR,'**','*.csv')

FORCE_SYNC        = False
MAX_ROWS_PER_FILE = 100_000   # cap stratified per file (kendali memori)
TARGET_TRAIN      = 100_000   # subsample train per klien agar seimbang
TEST_SIZE         = 0.30
SEED              = 42
np.random.seed(SEED)
print('Config OK | bucket =', S3_BUCKET, '| prefix =', S3_PREFIX)

## 1. Sync dataset 2017 & 2018 dari S3 (bila lokal kosong) + auto-unzip

In [ ]:
def has_csv(d): return len(glob.glob(os.path.join(d,'**','*.csv'),recursive=True))>0
def has_zip(d): return len(glob.glob(os.path.join(d,'**','*.zip'),recursive=True))>0
def s3_sync(src,dst):
    print('sync',src,'->',dst)
    subprocess.run(['aws','s3','sync',src,dst,'--only-show-errors'],check=True)
def unzip_all(d):
    for zp in glob.glob(os.path.join(d,'**','*.zip'),recursive=True):
        marker=zp+'.extracted'
        if os.path.exists(marker): continue
        print('  unzip',os.path.basename(zp))
        with zipfile.ZipFile(zp) as z: z.extractall(d)
        open(marker,'w').close()
def ensure_local(src,ldir,name):
    if (not FORCE_SYNC) and has_csv(ldir):
        n=len(glob.glob(os.path.join(ldir,'**','*.csv'),recursive=True))
        print('[%s] %d CSV sudah ada -> lewati download'%(name,n)); return
    s3_sync(src,ldir)
    if has_zip(ldir): unzip_all(ldir)
    n=len(glob.glob(os.path.join(ldir,'**','*.csv'),recursive=True))
    print('[%s] selesai: %d CSV siap'%(name,n))
ensure_local(S3_RAW_CIC2017, CIC2017_DIR, 'CIC2017')
ensure_local(S3_RAW_CIC2018, CIC2018_DIR, 'CIC2018')
assert has_csv(CIC2017_DIR), 'PRASYARAT GAGAL: tak ada CSV CIC2017 di %s'%CIC2017_DIR
assert has_csv(CIC2018_DIR), 'PRASYARAT GAGAL: tak ada CSV CIC2018 di %s'%CIC2018_DIR
print('dataset siap di lokal')

## 2. Util normalisasi kolom + label (kamus ALIAS 2018<->2017, reuse Paper 4)

In [ ]:
META_SUBSTR=['flow id','flowid','source ip','src ip','destination ip','dst ip',
             'source port','src port','destination port','dst port','protocol',
             'timestamp','unnamed']
NONFEATURE={'accuracy (%)','f1','f1-score (%)','mcc','precision','precision (%)',
            'recall (%)','model size (mb)','training (s)','inference/10k (s)','n_feat'}
# samakan penamaan rilis 2018 (singkat) ke rilis 2017 (panjang)
ALIAS={
 'tot fwd pkts':'total fwd packets','tot bwd pkts':'total backward packets',
 'totlen fwd pkts':'total length of fwd packets','totlen bwd pkts':'total length of bwd packets',
 'fwd pkt len max':'fwd packet length max','fwd pkt len min':'fwd packet length min',
 'fwd pkt len mean':'fwd packet length mean','fwd pkt len std':'fwd packet length std',
 'bwd pkt len max':'bwd packet length max','bwd pkt len min':'bwd packet length min',
 'bwd pkt len mean':'bwd packet length mean','bwd pkt len std':'bwd packet length std',
 'pkt len max':'max packet length','pkt len min':'min packet length',
 'pkt len mean':'packet length mean','pkt len std':'packet length std',
 'pkt len var':'packet length variance','pkt size avg':'average packet size',
 'fwd seg size avg':'avg fwd segment size','bwd seg size avg':'avg bwd segment size',
 'fwd seg size min':'min_seg_size_forward','fwd act data pkts':'act_data_pkt_fwd',
 'fwd header len':'fwd header length','bwd header len':'bwd header length',
 'fwd iat tot':'fwd iat total','bwd iat tot':'bwd iat total',
 'flow byts/s':'flow bytes/s','flow pkts/s':'flow packets/s',
 'fwd pkts/s':'fwd packets/s','bwd pkts/s':'bwd packets/s',
 'init fwd win byts':'init_win_bytes_forward','init bwd win byts':'init_win_bytes_backward',
 'subflow fwd pkts':'subflow fwd packets','subflow bwd pkts':'subflow bwd packets',
 'subflow fwd byts':'subflow fwd bytes','subflow bwd byts':'subflow bwd bytes',
 'fin flag cnt':'fin flag count','syn flag cnt':'syn flag count',
 'rst flag cnt':'rst flag count','psh flag cnt':'psh flag count',
 'ack flag cnt':'ack flag count','urg flag cnt':'urg flag count',
 'ece flag cnt':'ece flag count','cwe flag count':'cwe flag count',
 'fwd byts/b avg':'fwd avg bytes/bulk','fwd pkts/b avg':'fwd avg packets/bulk',
 'fwd blk rate avg':'fwd avg bulk rate','bwd byts/b avg':'bwd avg bytes/bulk',
 'bwd pkts/b avg':'bwd avg packets/bulk','bwd blk rate avg':'bwd avg bulk rate',
}
def norm_colname(c):
    c=str(c).replace('\ufeff','').strip(); c=' '.join(c.split()); c=c.lower()
    return ALIAS.get(c,c)
def find_label_col(cols):
    cand=[c for c in cols if 'label' in c]; return cand[0] if cand else cols[-1]
def to_binary(label):
    s=str(label).strip().lower(); return 0 if ('benign' in s or s=='normal' or s=='0') else 1
def is_meta(c): return any(m in c for m in META_SUBSTR) or c in NONFEATURE
def load_glob(pattern,name):
    files=sorted(glob.glob(pattern,recursive=True))
    files=[f for f in files if not any(seg in f.replace(chr(92),'/').lower()
           for seg in ['/results/','/result/','/artifacts/','/output/'])]
    assert files, 'Tak ada file dataset %s: %s'%(name,pattern)
    print('[%s] %d file dataset'%(name,len(files)))
    frames=[]
    for fp in files:
        df=pd.read_csv(fp,low_memory=False)
        df.columns=[norm_colname(c) for c in df.columns]
        lab=find_label_col(list(df.columns)); df=df.rename(columns={lab:'label'})
        if len(df)>MAX_ROWS_PER_FILE:
            frac=MAX_ROWS_PER_FILE/len(df)
            df=df.groupby('label',group_keys=False).apply(
                lambda g: g.sample(max(1,int(len(g)*frac)),random_state=SEED))
        frames.append(df); print('   %s: %d baris (cap)'%(os.path.basename(fp),len(df)))
    out=pd.concat(frames,ignore_index=True); del frames; gc.collect(); return out
print('Util OK (alias + exclude results/)')

## 3. Muat + bersihkan + label biner + irisan fitur (~76)

In [ ]:
cic17=load_glob(CIC2017_GLOB,'CIC2017'); cic18=load_glob(CIC2018_GLOB,'CIC2018')
print('raw shapes:',cic17.shape,cic18.shape)

def clean(df):
    y=df['label'].apply(to_binary).astype(int).values
    X=df.drop(columns=['label'])
    keep=[c for c in X.columns if not is_meta(c)]
    X=X[keep].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    X=X.dropna(axis=1,how='all')
    return X,y
X17,y17=clean(cic17); X18,y18=clean(cic18)
print('CIC2017:',X17.shape,'attack%=',round(100*y17.mean(),2))
print('CIC2018:',X18.shape,'attack%=',round(100*y18.mean(),2))

common=sorted(set(X17.columns)&set(X18.columns))
print('\nIRISAN fitur:',len(common))
only17=sorted(set(X17.columns)-set(X18.columns)); only18=sorted(set(X18.columns)-set(X17.columns))
print('Hanya 2017:',only17[:15],'...' if len(only17)>15 else '')
print('Hanya 2018:',only18[:15],'...' if len(only18)>15 else '')
X17=X17[common].copy(); X18=X18[common].copy()
# imputasi median per kolom (fit di masing-masing; normalisasi z-score dilakukan saat training)
X17=X17.fillna(X17.median(numeric_only=True)).fillna(0.0).astype(float)
X18=X18.fillna(X18.median(numeric_only=True)).fillna(0.0).astype(float)
FEATS=list(common)
print('ruang fitur near-domain:',len(FEATS))

## 4. Split 70/30 per klien + subsample train seimbang

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
def split_balance(X,y,name):
    Xtr,Xte,ytr,yte=train_test_split(X.values,y,test_size=TEST_SIZE,random_state=SEED,stratify=y)
    if len(Xtr)>TARGET_TRAIN:
        idx=resample(np.arange(len(Xtr)),n_samples=TARGET_TRAIN,replace=False,
                     stratify=ytr,random_state=SEED)
        Xtr=Xtr[idx]; ytr=ytr[idx]
    print('%s train %s pos=%.3f | test %s pos=%.3f'%(
        name,Xtr.shape,float(ytr.mean()),Xte.shape,float(yte.mean())))
    return Xtr,Xte,ytr,yte
X17tr,X17te,y17tr,y17te=split_balance(X17,y17,'CIC2017')
X18tr,X18te,y18tr,y18te=split_balance(X18,y18,'CIC2018')

## 5. Statistik per-klien (ANGKA NYATA) -> tabel jurnal

In [ ]:
rows=[]
def stat(name,y,scheme):
    n=len(y); atk=int(np.sum(y==1)); ben=int(np.sum(y==0))
    rows.append({'client':name,'total':n,'attack':atk,'benign':ben,
                 'pos_rate':round(atk/n,4) if n else 0.0,'scheme':scheme})
stat('Client-CIC2017 (train)',y17tr,'70/30 seed, subsample')
stat('Client-CIC2017 (test)', y17te,'held-out')
stat('Client-CIC2018 (train)',y18tr,'70/30 seed, subsample')
stat('Client-CIC2018 (test)', y18te,'held-out')
dfst=pd.DataFrame(rows); dfst.to_csv(os.path.join(OUTDIR,'neardomain_stats.csv'),index=False)
import IPython.display as ipd; ipd.display(dfst)
print('=== SEL 5 (statistik) SELESAI ===')

## 6. Simpan partisi (.npz) + schema + upload S3

Nama file mengikuti pola node EC2: `<tag>_train.npz`/`<tag>_test.npz` (key `X`,`y`,`feats`).

In [ ]:
def save_part(prefix,X,y):
    p=os.path.join(OUTDIR,prefix+'.npz')
    np.savez_compressed(p,X=np.asarray(X,float),y=np.asarray(y,int),feats=np.array(FEATS))
    print('saved',p,np.asarray(X).shape)
for nm,Xv,yv in [('cic2017_train',X17tr,y17tr),('cic2017_test',X17te,y17te),
                 ('cic2018_train',X18tr,y18tr),('cic2018_test',X18te,y18te)]:
    save_part(nm,Xv,yv)

schema={'config':'near-domain (CICFlowMeter 2017 vs 2018)','seed':SEED,
        'test_size':TEST_SIZE,'target_train':TARGET_TRAIN,'max_rows_per_file':MAX_ROWS_PER_FILE,
        'n_common_features':len(FEATS),'features':FEATS,
        'cic2017':{'train':int(len(y17tr)),'test':int(len(y17te)),'attack_pct':round(100*float(y17.mean()),2)},
        'cic2018':{'train':int(len(y18tr)),'test':int(len(y18te)),'attack_pct':round(100*float(y18.mean()),2)}}
json.dump(schema,open(os.path.join(OUTDIR,'neardomain_schema.json'),'w'),indent=2)
print('schema fitur =',len(FEATS))

try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for f in os.listdir(OUTDIR):
        if f.endswith('.npz') or f.endswith('.csv') or f.endswith('.json'):
            if f.startswith('cic2017'): sub='cic2017'
            elif f.startswith('cic2018'): sub='cic2018'
            else: sub='meta'
            s3.upload_file(os.path.join(OUTDIR,f),S3_BUCKET,'%s/data/%s/%s'%(S3_PREFIX,sub,f)); up+=1
    print('upload',up,'-> s3://%s/%s/data/'%(S3_BUCKET,S3_PREFIX))
except Exception as e:
    print('(upload S3 dilewati:',e,')')
print('=== SEL 6 (simpan + upload) SELESAI ===')

## 7. Catatan metodologi (untuk naskah jurnal)

- **Near-domain:** kedua klien diekstrak dengan ekstraktor SAMA (CICFlowMeter), jadi
  irisan fitur jauh lebih besar (~76) daripada konfig cross-domain dengan UNSW (9).
  Ini memungkinkan menguji FL pada 'jarak domain dekat'.
- **Penyelarasan fitur = irisan nama kolom** setelah normalisasi (kamus ALIAS
  menyamakan singkatan rilis 2018 ke nama panjang rilis 2017). Bukan SFM, bukan
  isi-nol; diperlakukan sebagai setup, konsisten dengan Paper 4 XAI.
- **Label biner:** Benign/normal = 0, selain itu = 1.
- **Normalisasi:** hanya imputasi median di sini; z-score **per-klien lokal**
  dilakukan di tahap training (konsisten protokol bebas-kebocoran jurnal).
- **Prefix S3 terpisah** (`federated_journal/neardomain/`) agar tak bercampur data survei.
- Angka attack% dan jumlah baris yang dilaporkan di paper diambil dari
  `neardomain_stats.csv` / `neardomain_schema.json` (hasil eksekusi nyata).